### Imports 

In [11]:
import os
import json
import statistics

### Average and Median RTTs for common and uncommon countries

In [14]:
def process_json_file(file_path, final_result, uncommon_anchors_file, common_anchors_file, anchor_probe_info):
    with open(file_path, 'r') as file:
        data = json.load(file)
    with open(common_anchors_file, 'r') as file:
        common_anchors = json.load(file)
    with open(uncommon_anchors_file, 'r') as file:
        uncommon_anchors = json.load(file)
    
    for measurement in data:
        prb_id = measurement.get('prb_id')
        avg_value = measurement.get('avg')
        min_value = measurement.get('min')
        max_value = measurement.get('max')

        # Check if prb_id is present and corresponds to anchor_id
        for anchor_id, stored_prb_id in anchor_probe_info.items():
            if stored_prb_id == prb_id:
                # Classify anchor as uncommon or common
                if anchor_id in uncommon_anchors.keys():
                    category = 'uncommon'
                elif anchor_id in common_anchors.keys():
                    category = 'common'
                else:
                    continue  # Skip if anchor is not in either category

                # Update final result based on the category
                if min_value > 0:
                    final_result['median'][category]['min'].append(min_value)
                    final_result['average'][category]['min'].append(min_value)
                    final_result['median']['all']['min'].append(min_value)
                    final_result['average']['all']['min'].append(min_value)
                if avg_value > 0:
                    final_result['median'][category]['avg'].append(avg_value)
                    final_result['average'][category]['avg'].append(avg_value)
                    final_result['median']['all']['avg'].append(avg_value)
                    final_result['average']['all']['avg'].append(avg_value)
                if max_value > 0:
                    final_result['median'][category]['max'].append(max_value)
                    final_result['average'][category]['max'].append(max_value)
                    final_result['median']['all']['max'].append(max_value)
                    final_result['average']['all']['max'].append(max_value)

                # break  # Break the loop once the corresponding anchor is found

In [ ]:
input_folder = '<data/raw>'
output_file = '<OUTPUT_STATS_JSON_PATH>'
uncommon_anchors_file = 'data/vantage_points/uncommon_countries_anchors.json'
common_anchors_file = 'data/vantage_points/anchors_only_in_top_common_countries.json'
anchor_probe_info_file = 'data/anchor_probe_info/anchor_probe_info.json'

final_result = {'average':{'common': {'min': [], 'avg': [], 'max': []},
                            'uncommon': {'min': [], 'avg': [], 'max': []},
                            'all': {'min': [], 'avg': [], 'max': []},
                            },
                            
                'median':{'common': {'min': [], 'avg': [], 'max': []},
                            'uncommon': {'min': [], 'avg': [], 'max': []},
                            'all' : {'min': [], 'avg': [], 'max': []},
                            }}
copy_final_result = final_result
# Load anchor classifications
with open(uncommon_anchors_file, 'r') as uncommon_anchors_file:
    uncommon_anchors = json.load(uncommon_anchors_file)

with open(common_anchors_file, 'r') as common_anchors_file:
    common_anchors = json.load(common_anchors_file)

# Load probe to anchor mappings
with open(anchor_probe_info_file, 'r') as anchor_probe_info_file:
    anchor_probe_info = json.load(anchor_probe_info_file)

for filename in os.listdir(input_folder):
    if filename.endswith('.json'):
        file_path = os.path.join(input_folder, filename)
        process_json_file(file_path, final_result, uncommon_anchors, common_anchors, anchor_probe_info)

# Calculate medians
for category in final_result['median']:
    for metric in final_result['average'][category]:
        if final_result['average'][category][metric]:
            final_result['average'][category][metric] = statistics.median(final_result['average'][category][metric])
        if final_result['median'][category][metric]:
                final_result['median'][category][metric] = sum(final_result['median'][category][metric]) / len(final_result['median'][category][metric])
with open(output_file, 'w') as output_file:
    json.dump(final_result, output_file, indent=2)


TypeError: unhashable type: 'dict'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_cdf(final_result, key):
    value = final_result
    plt.figure(figsize=(18, 9))
    value.sort()
    cumulative_prob = np.arange(1, len(value) + 1) / len(value)

    plt.plot(value, cumulative_prob, marker='x', linestyle='-', color='k', mec='r')
    plt.title(f'CDF of {key}')
    plt.xlabel(f'Values of {key}')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.savefig(f'CDF_{key}.png')

def plot_cdf_log(final_result, key):
    value = final_result
    plt.figure(figsize=(18, 9))
    value.sort()
    cumulative_prob = np.arange(1, len(value) + 1) / len(value)

    plt.plot(np.log10(value), cumulative_prob, marker='x', linestyle='-', color='k', mec='r')
    plt.title(f'CDF of log of {key}')
    plt.xlabel(f'Values of {key}')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.savefig(f'CDF_log_{key}.png')

In [ ]:
plot_cdf_log(final_result['average']['all']['min'], 'average_all_min')

### Finding interesting measurements

### Number of Unique Destination IP addresses

In [1]:
import json

def process_json_file(file_path, unique_dst_addrs, unique_src_addrs):
    with open(file_path, 'r') as file:
        data = json.load(file)

    for measurement in data:
        dst_addr = measurement.get('dst_addr')
        src_addr = measurement.get('src_addr')
        if dst_addr and dst_addr not in unique_dst_addrs:
            unique_dst_addrs.add(dst_addr)
        if src_addr and src_addr not in unique_src_addrs:
            unique_src_addrs.add(src_addr)

In [ ]:
import os
import json

input_folder = "<data/raw>"
# json_output_file = "<UNIQUE_DST_ADDRS_JSON_PATH>"
txt_dst_file = "<UNIQUE_DST_ADDRS_TXT_PATH>"
txt_src_file = "<UNIQUE_SRC_ADDRS_TXT_PATH>"

unique_dst_addrs = set()
unique_src_addrs = set()


for filename in os.listdir(input_folder):
    if filename.endswith(".json"):
        file_path = os.path.join(input_folder, filename)
        process_json_file(file_path, unique_dst_addrs, unique_src_addrs)

# result = {"unique_dst_addrs": list(unique_dst_addrs)}
print("Unique destination addresses found:", len(unique_dst_addrs))
print("Unique source addresses found:", len(unique_src_addrs))

# Save as JSON
# with open(json_output_file, "w") as f_json:
    # json.dump(result, f_json, indent=2)

# Save as TXT
with open(txt_dst_file, "w") as f_txt:
    for addr in sorted(unique_dst_addrs):
        f_txt.write(addr + "\n")
with open(txt_src_file, "w") as f_txt:
    for addr in sorted(unique_src_addrs):
        f_txt.write(addr + "\n")


Unique destination addresses found: 13934
Unique source addresses found: 69


### CDFs of metrics


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

def process_json_file(file_path, final_result):
    
    return final_result

def main():
    input_folder = '<data/raw>'
    output_file = '<AVERAGES_OUTPUT_JSON_PATH>'

    final_result = {'min':[],
                    'avg':[],
                    'max':[]}

    for filename in os.listdir(input_folder):
        if filename.endswith('.json'):
            file_path = os.path.join(input_folder, filename)
            with open(file_path, 'r') as file:
                data = json.load(file)

            # value_dict = {}
            # result_dict= {}
            # global_msm_id = 0
            for measurement in data:
                # src_addr = measurement.get('src_addr')
                avg_value = measurement.get('avg')
                min_value = measurement.get('min')
                max_value =measurement.get('max')
                # msm_id = measurement.get('msm_id')
                if(min_value>0):
                    final_result['min'].append(min_value)
                if(avg_value>0):
                    final_result['avg'].append(avg_value)
                if(max_value>0):
                    final_result['max'].append(max_value)

    # with open(output_file, 'w') as output_file:
    #     json.dump(final_result, output_file, indent=2)
    
    # for key, value in final_result.items():
    #     print("median of ", key, np.average(value))
    
    # for key, value in final_result.items():
    #     plt.figure(figsize=(18, 9))
    #     plt.hist(value, bins=200, color='blue', edgecolor='black', alpha=0.7)
    #     plt.title(f'Histogram of {key}')
    #     plt.xlabel(f'Values of {key}')
    #     plt.ylabel('Frequency')
    #     plt.grid(True)
    #     # plt.show()
    #     plt.savefig(f'PDF_{key}.png')
    # for key, value in final_result.items():
    #     plt.figure(figsize=(18, 9))
    #     plt.hist(np.log10(value), bins=200, color='blue', edgecolor='black', alpha=0.7)
    #     plt.title(f'Histogram of log of {key}')
    #     plt.xlabel(f'Values of {key}')
    #     plt.ylabel('Frequency')
    #     plt.grid(True)
    #     # plt.show()
    #     plt.savefig(f'PDF_log_{key}.png')

    
    for key, value in final_result.items():
        plt.figure(figsize=(18, 9))
        value.sort()
        cumulative_prob = np.arange(1, len(value) + 1) / len(value)

        # Plot the CDF
        # plt.figure(figsize=(18, 9))  # Adjust the width and height as needed
        plt.plot(value, cumulative_prob, marker='x', linestyle='-', color='k', mec='r')
        # plt.hist(value, bins=200, color='blue', edgecolor='black', alpha=0.7)
        plt.title(f'CDF of {key}')
        plt.xlabel(f'Values of {key}')
        plt.ylabel('Frequency')
        plt.grid(True)
        # plt.show()
        plt.savefig(f'CDF_{key}.png')
    for key, value in final_result.items():
        plt.figure(figsize=(18, 9))
        value.sort()
        cumulative_prob = np.arange(1, len(value) + 1) / len(value)

        # Plot the CDF
        # plt.figure(figsize=(18, 9))  # Adjust the width and height as needed
        plt.plot(np.log10(value), cumulative_prob, marker='x', linestyle='-', color='k', mec='r')
        # plt.hist(np.log10(value), bins=200, color='blue', edgecolor='black', alpha=0.7)
        plt.title(f'CDF of log of {key}')
        plt.xlabel(f'Values of {key}')
        plt.ylabel('Frequency')
        plt.grid(True)
        # plt.show()
        plt.savefig(f'CDF_log_{key}.png')


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv(
    '<MEDIAN_PLOT_DATA_CSV_PATH>', keep_default_na=False, na_values=["nan"], low_memory=False
)

    #     df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')
    #     df['Day'] = pd.to_datetime(df['Day'])
df["Destination ASN"] = df["Destination ASN"].astype(str)
df["Source ASN"] = df["Source ASN"].astype(str)

# Ensure the 'Spike' column is boolean
df['Spike'] = df['Spike (Normalized >= 10)'].astype(bool)

# Function to calculate spike rate for a group
def compute_spike_rate(group):
    total = len(group)
    spikes = group['Spike'].sum()
    return 100 * spikes / total if total > 0 else 0

# Group and sort by Source Continent
spike_rate_src_continent = (
    df.groupby('Source Continent')
    .apply(compute_spike_rate)
    .reset_index(name='Spike Rate (%)')
    .sort_values(by='Spike Rate (%)', ascending=False)
)

# Group and sort by Destination Continent
spike_rate_dst_continent = (
    df.groupby('Destination Continent')
    .apply(compute_spike_rate)
    .reset_index(name='Spike Rate (%)')
    .sort_values(by='Spike Rate (%)', ascending=False)
)

# Group and sort by Source ASN
spike_rate_src_asn = (
    df.groupby('Source ASN')
    .apply(compute_spike_rate)
    .reset_index(name='Spike Rate (%)')
    .sort_values(by='Spike Rate (%)', ascending=False)
)

# Group and sort by Destination ASN
spike_rate_dst_asn = (
    df.groupby('Destination ASN')
    .apply(compute_spike_rate)
    .reset_index(name='Spike Rate (%)')
    .sort_values(by='Spike Rate (%)', ascending=False)
)

# Print results
print("Spike Rate by Source Continent (Descending):")
print(spike_rate_src_continent)

print("\nSpike Rate by Destination Continent (Descending):")
print(spike_rate_dst_continent)

print("\nSpike Rate by Source ASN (Descending):")
print(spike_rate_src_asn)

print("\nSpike Rate by Destination ASN (Descending):")
print(spike_rate_dst_asn)


/tmp/ipykernel_782/1151214927.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_spike_rate)
/tmp/ipykernel_782/1151214927.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_spike_rate)
/tmp/ipykernel_782/1151214927.py:41: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping 

Spike Rate by Source Continent (Descending):
  Source Continent  Spike Rate (%)
5               SA        6.423891
0               AF        5.144496
3               NA        3.695984
4               OC        3.287793
2               EU        3.202446
1               AS        2.413543
6          Unknown        0.000000

Spike Rate by Destination Continent (Descending):
  Destination Continent  Spike Rate (%)
5                    SA       11.940836
0                    AF        5.036894
4                    OC        4.519372
1                    AS        4.262539
2                    EU        4.217089
3                    NA        2.175006

Spike Rate by Source ASN (Descending):
   Source ASN  Spike Rate (%)
14       2716       10.420184
1       12303        7.297476
6      202422        6.068920
21      42961        5.540922
3      141235        5.085222
18       3491        4.925740
5       15133        3.795689
25       9268        3.089888
0       12008        2.742485
7   

/tmp/ipykernel_782/1151214927.py:49: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_spike_rate)


In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv(
    '<MEDIAN_PLOT_DATA_CSV_PATH>',
    keep_default_na=False,
    na_values=["nan"],
    low_memory=False
)

# Ensure 'Spike' is boolean
df['Spike'] = df['Spike (Normalized >= 10)'].astype(bool)

# Combine ASN and continent roles
asn_records = pd.concat([
    df[['Source ASN', 'Spike']].rename(columns={'Source ASN': 'ASN'}),
    df[['Destination ASN', 'Spike']].rename(columns={'Destination ASN': 'ASN'})
])

continent_records = pd.concat([
    df[['Source Continent', 'Spike']].rename(columns={'Source Continent': 'Continent'}),
    df[['Destination Continent', 'Spike']].rename(columns={'Destination Continent': 'Continent'})
])

# Compute combined spike rates
asn_spike_rate = (
    asn_records.groupby('ASN')['Spike']
    .agg(['count', 'sum'])
    .rename(columns={'count': 'Total', 'sum': 'Spikes'})
)
asn_spike_rate['Spike Rate (%)'] = 100 * asn_spike_rate['Spikes'] / asn_spike_rate['Total']
asn_spike_rate = asn_spike_rate.sort_values(by='Spike Rate (%)', ascending=False).reset_index()

continent_spike_rate = (
    continent_records.groupby('Continent')['Spike']
    .agg(['count', 'sum'])
    .rename(columns={'count': 'Total', 'sum': 'Spikes'})
)
continent_spike_rate['Spike Rate (%)'] = 100 * continent_spike_rate['Spikes'] / continent_spike_rate['Total']
continent_spike_rate = continent_spike_rate.sort_values(by='Spike Rate (%)', ascending=False).reset_index()

# Print results
print("Combined Spike Rate by ASN (source or destination):")
print(asn_spike_rate.head(10))  # Display top 10 ASNs

print("\nCombined Spike Rate by Continent (source or destination):")
print(continent_spike_rate)


Combined Spike Rate by ASN (source or destination):
      ASN  Total  Spikes  Spike Rate (%)
0   37100     17      17      100.000000
1   23844      6       6      100.000000
2    4637     12      12      100.000000
3     174     21      16       76.190476
4   15802      4       3       75.000000
5    5408    164     119       72.560976
6  213736     94      57       60.638298
7    7195     35      17       48.571429
8  139057    401     162       40.399002
9  135391     16       6       37.500000

Combined Spike Rate by Continent (source or destination):
  Continent   Total  Spikes  Spike Rate (%)
0        SA  107803    8764        8.129644
1        AF   60883    3112        5.111443
2        OC   65154    2419        3.712742
3        EU  297239   10733        3.610899
4        AS  362100   10750        2.968793
5        NA  553233   13974        2.525880
6   Unknown   10620       0        0.000000


In [3]:
continent_spike_rate = continent_spike_rate.sort_values(by='Spikes', ascending=False).reset_index()
asn_spike_rate = asn_spike_rate.sort_values(by='Spikes', ascending=False).reset_index()



# Print results
print("Combined Spike Rate by ASN (source or destination):")
print(asn_spike_rate.head(10))  # Display top 10 ASNs

print("\nCombined Spike Rate by Continent (source or destination):")
print(continent_spike_rate)

Combined Spike Rate by ASN (source or destination):
   index     ASN   Total  Spikes  Spike Rate (%)
0     27    3491  148262    7303        4.925740
1     22  202422  117006    7101        6.068920
2     16   16625   66881    5490        8.208609
3     33   54113  159257    4917        3.087462
4     23   16509   79301    4686        5.909131
5     31   15169  128262    4494        3.503766
6     17   20940   47066    3522        7.483109
7     34   12008   73984    2029        2.742485
8     29   15133   53350    2025        3.795689
9     41   20473  113715    1787        1.571473

Combined Spike Rate by Continent (source or destination):
   index Continent   Total  Spikes  Spike Rate (%)
0      5        NA  553233   13974        2.525880
1      4        AS  362100   10750        2.968793
2      3        EU  297239   10733        3.610899
3      0        SA  107803    8764        8.129644
4      1        AF   60883    3112        5.111443
5      2        OC   65154    2419        3.

In [ ]:
import os 
import json
from collections import defaultdict 

with open('data/vantage_points/all_anchors_details.json', 'r') as file:
    anchors = json.load(file)

    countries = set()
    asn = set()

    for key,value in anchors.items():
        countries.add(value['country'])
        asn.add(value['as'])
        # print(key, value['country'])

print("Countries: ", len(countries))
print("ASNs: ", len(asn))

Countries:  34
ASNs:  27
